In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
logging.basicConfig(level=logging.ERROR, format='%(levelname)s: %(message)s')

import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf # type: ignore

from meridian.model import model
from meridian.model import spec
from meridian.analysis import optimizer
from meridian.analysis import analyzer

from meridian.planner.flex_budget_planner import FlexibleBudgetPlanner, CompareOptimizedVsNonOptimized
from meridian.analysis.optimizer import OptimizationResults


In [4]:
opt_period = {
    'Allergan': {'start_date': '2024-07-06', 'end_date': '2025-06-28'},
    'Burger_King': {'start_date': '2024-07-06', 'end_date': '2025-06-28'},
    'Audi': {'start_date': '2024-07-06', 'end_date': '2025-06-28'},
    'Samsung_US_Starcom': {'start_date': '2024-07-06', 'end_date': '2025-06-28'},
    'MRG_Chevy_LMA': {'start_date': '2024-07-06', 'end_date': '2025-06-28'},
    'Mazda': {'start_date': '2024-07-06', 'end_date': '2025-06-28'},
    'Popeyes': {'start_date': '2024-07-06', 'end_date': '2025-06-28'}
  }

In [ ]:
# Optimizer input excel file path
home_dir = '/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer'

In [7]:
# Configuration for input excel file
input_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',
  'population_col': 'population',

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Display_impression', 'TV_impression', 'Video_impression'],
  'media_spend_cols': ['Display_spend', 'TV_spend', 'Video_spend'],
  'media_channels': ['Display', 'TV', 'Video']

}

# optimization config
optimization_config = {
  'fixed_budget': True,
  'use_kpi': True,

  # spend constraints
  'spend_constraint_lower': {  # (1 - value)% of historical
    'Display': 0.3,
    'TV': 0.3,
    'Video': 0.3,
  },

  'spend_constraint_upper': {  # (1 + value)% of historical
  'Display': 0.3,
  'TV': 0.3,
  'Video': 0.3,
  }
}

In [8]:
advertisers = ['Popeyes', 'Allergan', 'MRG_Chevy_LMA', 'Mazda', 'Samsung_US_Starcom', 'Audi', 'Burger_King']

opt_results_all = {}
total_list = []
channel_list = []
for model_type in ['default', 'champ']:
  for client in advertisers:
    print(f"...... Optimizing {client} based on {model_type} model.......", sep='\n')
    opt_results_all[client] = {}

    # get path to the input file
    input_file_path = f'{home_dir}/optimizer_inputs/{model_type}/{client}_optimizer_input_{model_type}.xlsx'
    if not os.path.exists(input_file_path):
      raise FileNotFoundError(f'File not found: {input_file_path}')

    # optimizer config
    opt_config = optimization_config.copy()
    opt_config['start_date'] = opt_period[client]['start_date']
    opt_config['end_date'] = opt_period[client]['end_date']

    # call the planner
    planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
    opt_results = planner.optimize(opt_config)

    # summarize optimized vs non-optimized
    opt_results_all[client]['opt_results'] = opt_results
    compare_opt_vs_nonopt = CompareOptimizedVsNonOptimized(opt_results)
    total_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_total_level_comparison()
    channel_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_channel_level_comparison()

    # append to list
    total_opt_vs_nonopt_df.insert(0, 'client', client)
    total_opt_vs_nonopt_df.insert(0, 'model_type', model_type)
    total_list.append(total_opt_vs_nonopt_df)

    channel_opt_vs_nonopt_df.insert(0, 'client', client)
    channel_opt_vs_nonopt_df.insert(0, 'model_type', model_type)
    channel_list.append(channel_opt_vs_nonopt_df)

    # export summary as html
    opt_results.output_optimization_summary(f'{client}_{model_type}.html', f'{home_dir}/optimizer_outputs')


# combine all the dataframes
total_df = pd.concat(total_list)
channel_df = pd.concat(channel_list)
total_df.to_csv(f'{home_dir}/optimizer_outputs/total_level_comparison_summary.csv', index=False)
channel_df.to_csv(f'{home_dir}/optimizer_outputs/channel_level_comparison_summary.csv', index=False)


...... Optimizing Popeyes based on default model.......


I0000 00:00:1758740611.615723  457998 service.cc:148] XLA service 0x129f9b310 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758740611.615757  457998 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1758740611.625237  457998 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-09-24 14:03:32.436704: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


...... Optimizing Allergan based on default model.......
...... Optimizing MRG_Chevy_LMA based on default model.......
...... Optimizing Mazda based on default model.......
...... Optimizing Samsung_US_Starcom based on default model.......
...... Optimizing Audi based on default model.......
...... Optimizing Burger_King based on default model.......
...... Optimizing Popeyes based on champ model.......
...... Optimizing Allergan based on champ model.......
...... Optimizing MRG_Chevy_LMA based on champ model.......
...... Optimizing Mazda based on champ model.......
...... Optimizing Samsung_US_Starcom based on champ model.......
...... Optimizing Audi based on champ model.......
...... Optimizing Burger_King based on champ model.......


In [9]:
total_df.query('model_type == "default"')

,model_type,client,start_date,end_date,optimized_budget,optimized_total_incremental_outcome,optimized_total_cpa,nonoptimized_budget,nonoptimized_total_incremental_outcome,nonoptimized_total_cpa,budget_change,outcome_change,cpa_change
0,default,Popeyes,2024-07-06,2025-06-28,28440000.0,4.512272e+08,0.063028,28430000.0,444227808.0,0.063999,0.000352,0.015756,-0.015166
0,default,Allergan,2024-07-06,2025-06-28,17740000.0,2.562014e+08,0.069242,17730000.0,253150720.0,0.070037,0.000564,0.012051,-0.011350
0,default,MRG_Chevy_LMA,2024-07-06,2025-06-28,29660000.0,1.318339e+08,0.224980,29650000.0,128638168.0,0.230491,0.000337,0.024843,-0.023912
0,default,Mazda,2024-07-06,2025-06-28,93440000.0,1.019889e+09,0.091618,91990000.0,999175040.0,0.092066,0.015763,0.020731,-0.004867
0,default,Samsung_US_Starcom,2024-07-06,2025-06-28,134200000.0,5.387439e+08,0.249098,133800000.0,527305216.0,0.253743,0.002990,0.021693,-0.018306
0,default,Audi,2024-07-06,2025-06-28,27680000.0,8.598907e+07,0.321901,27670000.0,83044912.0,0.333193,0.000361,0.035453,-0.033890
0,default,Burger_King,2024-07-06,2025-06-28,144800000.0,1.234113e+08,1.173313,144700000.0,123086328.0,1.175598,0.000691,0.002640,-0.001944


In [10]:
total_df.query('model_type == "champ"')

,model_type,client,start_date,end_date,optimized_budget,optimized_total_incremental_outcome,optimized_total_cpa,nonoptimized_budget,nonoptimized_total_incremental_outcome,nonoptimized_total_cpa,budget_change,outcome_change,cpa_change
0,champ,Popeyes,2024-07-06,2025-06-28,28440000.0,263236752.0,0.108040,28430000.0,240858368.0,0.118036,0.000352,0.092911,-0.084691
0,champ,Allergan,2024-07-06,2025-06-28,17740000.0,37729144.0,0.470194,17730000.0,32425034.0,0.546800,0.000564,0.163581,-0.140099
0,champ,MRG_Chevy_LMA,2024-07-06,2025-06-28,29660000.0,63815640.0,0.464776,29650000.0,61056036.0,0.485619,0.000337,0.045198,-0.042921
0,champ,Mazda,2024-07-06,2025-06-28,93440000.0,775468544.0,0.120495,91990000.0,740011648.0,0.124309,0.015763,0.047914,-0.030681
0,champ,Samsung_US_Starcom,2024-07-06,2025-06-28,133900000.0,136177184.0,0.983278,133800000.0,116843008.0,1.145126,0.000747,0.165471,-0.141337
0,champ,Audi,2024-07-06,2025-06-28,27680000.0,30746008.0,0.900279,27670000.0,27730836.0,0.997806,0.000361,0.108730,-0.097741
0,champ,Burger_King,2024-07-06,2025-06-28,144800000.0,79126984.0,1.829970,144700000.0,78376160.0,1.846225,0.000691,0.009580,-0.008804


In [32]:
# ratio between 'total' optimized spend and non-optimized spend
total_historical_budget = nonopt_attrs['budget']
total_optimized_budget = opt_attrs['budget']

nonoptimized_outcome = nonopt_attrs['total_incremental_outcome']
optimized_outcome = opt_attrs['total_incremental_outcome']

print(f'ratio: {total_optimized_budget:.2f} / {total_historical_budget:.2f} = {total_optimized_budget / total_historical_budget:.2f}')
print(f'ratio: {optimized_outcome.sum() / nonoptimized_outcome.sum():.2f}')


ratio: 40680000.00 / 40670000.00 = 1.00
ratio: 1.12


In [34]:
import dataclasses
from meridian.analysis.optimizer import _exceeds_optimization_constraints, FixedBudgetScenario, FlexibleBudgetScenario

if optimization_config['fixed_budget']:
  scenario = FixedBudgetScenario(total_budget=np.sum(total_historical_budget))
else:
  scenario = FlexibleBudgetScenario(target_metric='roi', target_value=optimization_config['target_roi'])

spend_grid = opt_results.optimization_grid.spend_grid
incremental_outcome_grid = opt_results.optimization_grid.incremental_outcome_grid

1. Old Optimize Method

In [ ]:
old_optimize_spend = opt_results.optimization_grid._grid_search(spend_grid, incremental_outcome_grid, scenario)
print(old_optimize_spend.values.sum())
old_optimize_spend.to_dataframe().reset_index()

Optimization constraints exceeded
28470000


,channel,grid_spend_index,spend_grid
0,Display,0,3070000
1,TV,0,22860000
2,Video,0,2540000


In [37]:
old_optimize_spend.values.sum() / total_historical_budget

np.float64(0.7000245881485124)

2. New Optimize Method

In [105]:
new_optimize_spend = opt_results.optimization_grid._grid_search_iterative(spend_grid, incremental_outcome_grid, scenario)
print(new_optimize_spend.sum())
print(new_optimize_spend)

81350000
[ 8780000 65320000  7250000]


In [106]:
opt_df

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Display,8780000,0.107929,3432334.5,0.001101,0.390926,0.479243,2.558026,mean
1,TV,65320000,0.802950,692399680.0,0.218648,10.600118,9.013790,0.094339,mean
2,Video,7250000,0.089121,21998338.0,0.064187,3.034254,4.078759,0.329570,mean


New Optimize Method Manual

In [16]:
from meridian.analysis.optimizer import _exceeds_optimization_constraints, FixedBudgetScenario, FlexibleBudgetScenario

if optimization_config['fixed_budget']:
  scenario = FixedBudgetScenario(total_budget=np.sum(total_historical_budget))

spend_grid = opt_results.optimization_grid.spend_grid
incremental_outcome_grid = opt_results.optimization_grid.incremental_outcome_grid

In [17]:
# Convert xr.DataArray to numpy arrays for processing
spend_grid_np = spend_grid.values
outcome_grid_np = incremental_outcome_grid.values

# Step 1: Initialize with first row (minimum spend levels)
spend = spend_grid_np[0, :].copy()
incremental_outcome = outcome_grid_np[0, :].copy()


In [24]:
spend_grid_np.shape

(1959, 3)

In [31]:
# within for loop
i = 1
spend_delta = spend_grid_np[i, :] - spend
outcome_delta = outcome_grid_np[i, :] - incremental_outcome

print(spend_delta)
print(outcome_delta)

[10000. 10000. 10000.]
[  3952.5625 132528.      29158.    ]


In [36]:
roi = np.round(tf.math.divide_no_nan(outcome_delta, spend_delta), 8)
print(roi)
np.argmax(roi)

[ 0.39525625 13.2528      2.9158    ]


np.int64(1)

In [37]:
# within for loop
i = 1958
spend_delta = spend_grid_np[i, :] - spend
outcome_delta = outcome_grid_np[i, :] - incremental_outcome

print(spend_delta)
print(outcome_delta)

[      nan 19580000.       nan]
[           nan 2.91119992e+08            nan]


In [38]:
# Handle divide by zero and calculate ROI
roi = np.divide(
    outcome_delta,
    spend_delta,
    out=np.zeros_like(outcome_delta),
    where=spend_delta != 0
)
roi

array([        nan, 14.86823248,         nan])

In [42]:
valid_roi_mask = ~np.isnan(roi) & (spend_delta > 0)
masked_roi = np.where(valid_roi_mask, roi, -np.inf)
max_roi_channel = np.argmax(masked_roi)
print(max_roi_channel)

1


In [48]:
not np.any(roi)

False

In [45]:
roi[np.nanargmax(roi)]

np.float64(14.868232482124617)

In [29]:
np.round(tf.math.divide_no_nan(outcome_delta, spend_delta), 8)

array([        nan, 14.86823248,         nan])

In [23]:
spend_grid

<xarray.DataArray 'spend_grid' (grid_spend_index: 1959, channel: 3)> Size: 47kB
array([[ 3070000., 22860000.,  2540000.],
       [ 3080000., 22870000.,  2550000.],
       [ 3090000., 22880000.,  2560000.],
       ...,
       [      nan, 42420000.,       nan],
       [      nan, 42430000.,       nan],
       [      nan, 42440000.,       nan]])
Coordinates:
  * grid_spend_index  (grid_spend_index) int64 16kB 0 1 2 3 ... 1956 1957 1958
  * channel           (channel) object 24B 'Display' 'TV' 'Video'

In [62]:
spend_grid = opt_results.optimization_grid.spend_grid
incremental_outcome_grid = opt_results.optimization_grid.incremental_outcome_grid

In [63]:
spend_grid_bkp = spend_grid.copy()

In [64]:
spend = spend_grid[0, :].copy()
incremental_outcome = incremental_outcome_grid[0, :].copy()
spend_grid = spend_grid[1:, :]
incremental_outcome_grid = incremental_outcome_grid[1:, :]
iterative_roi_grid = np.round(
    tf.math.divide_no_nan(
        incremental_outcome_grid - incremental_outcome, spend_grid - spend
    ),
    decimals=8,
)


In [65]:
iterative_roi_grid[:5, :]

array([[ 0.40236875, 12.8864    ,  2.93505   ],
       [ 0.40275625, 12.8888    ,  2.938775  ],
       [ 0.40316667, 12.8936    ,  2.94263333],
       [ 0.40356562, 12.8938    ,  2.9463125 ],
       [ 0.40394875, 12.89584   ,  2.95006   ]])

In [66]:
iterative_roi_grid.shape

(1958, 3)

In [67]:
np.isnan(iterative_roi_grid).all()

np.False_

In [68]:
point = np.unravel_index(np.nanargmax(iterative_roi_grid), iterative_roi_grid.shape)
point

(np.int64(1710), np.int64(1))

In [69]:
iterative_roi_grid[point[0]]

array([       nan, 14.6334111,        nan])

In [70]:
row_idx = point[0]
media_idx = point[1]

In [71]:
spend[media_idx] = spend_grid[row_idx, media_idx]
incremental_outcome[media_idx] = incremental_outcome_grid[row_idx, media_idx]

In [72]:
roi_grid_point = iterative_roi_grid[row_idx, media_idx]
roi_grid_point

np.float64(14.6334111)

In [ ]:
import dataclasses
from meridian.analysis.optimizer import _exceeds_optimization_constraints, FixedBudgetScenario
scenario = FixedBudgetScenario
if isinstance(scenario, FixedBudgetScenario):
  scenario = dataclasses.replace(
      scenario, total_budget=np.sum(total_historical_budget)
  )

_exceeds_optimization_constraints(
          spend=spend,
          incremental_outcome=incremental_outcome,
          roi_grid_point=roi_grid_point,
          scenario=scenario,
      )

AttributeError: type object 'FixedBudgetScenario' has no attribute 'target_value'

In [45]:
np.random.seed(42)
samples = np.random.normal(loc=10, scale=0.2, size=20)
sample_array = np.reshape(samples, (4, 5))
print(sample_array)
np.unravel_index(np.nanargmax(sample_array), sample_array.shape)

[[10.09934283  9.97234714 10.12953771 10.30460597  9.95316933]
 [ 9.95317261 10.31584256 10.15348695  9.90610512 10.10851201]
 [ 9.90731646  9.90685405 10.04839245  9.61734395  9.65501643]
 [ 9.88754249  9.79743378 10.06284947  9.81839518  9.71753926]]


(np.int64(1), np.int64(1))

In [38]:
np.nanargmax(iterative_roi_grid)

np.int64(1)

In [13]:
opt_attrs

{'start_date': '2024-01-06',
 'end_date': '2025-06-28',
 'budget': np.float32(28470000.0),
 'profit': np.float32(104569030.0),
 'total_incremental_outcome': np.float32(133039030.0),
 'total_roi': np.float32(4.672955),
 'total_cpik': np.float32(0.21399735),
 'is_revenue_kpi': False,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [12]:
nonopt_attrs

{'start_date': '2024-01-06',
 'end_date': '2025-06-28',
 'budget': np.float32(40670000.0),
 'profit': np.float32(237170190.0),
 'total_incremental_outcome': np.float32(277840200.0),
 'total_roi': np.float32(6.831576),
 'total_cpik': np.float32(0.14637911),
 'is_revenue_kpi': False,
 'confidence_level': 0.9,
 'use_historical_budget': True}

In [21]:
historical_budget = nonopt_attrs['budget']
optimized_budget = opt_attrs['budget']
print(f'ratio: {optimized_budget:.2f} / {historical_budget:.2f} = {optimized_budget / historical_budget:.2f}')

ratio: 28470000.00 / 40670000.00 = 0.70


In [9]:
opt_df = opt_results.optimized_data.sel(metric='mean').to_dataframe().reset_index()
nonopt_df = opt_results.nonoptimized_data.sel(metric='mean').to_dataframe().reset_index()
opt_df['spend'].sum() / nonopt_df['spend'].sum()

np.float64(0.7000245881485124)

In [10]:
opt_df['spend'].values / opt_results.optimization_grid.historical_spend

array([0.69923134, 0.70009164, 0.70047396])

In [36]:
opt_df['spend'] / nonopt_df['spend']

0    0.700000
1    0.700000
2    0.699531
Name: spend, dtype: float64

In [37]:
opt_df['incremental_outcome'] / nonopt_df['incremental_outcome']

0    0.530360
1    0.489592
2    0.544670
Name: incremental_outcome, dtype: float32

In [24]:
opt_df

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Display,1890000,0.094975,2.951308e+05,0.000521,0.156154,0.287054,6.403939,mean
1,TV,16520000,0.830151,1.117774e+08,0.134829,6.766186,15.234431,0.147794,mean
2,Video,1490000,0.074874,2.688453e+06,0.035596,1.804331,3.189530,0.554222,mean


In [25]:
nonopt_df

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Display,2700000,0.094970,556473.0,0.000688,0.206101,0.354627,4.851987,mean
1,TV,23600000,0.830109,228307392.0,0.192773,9.674042,16.821016,0.103369,mean
2,Video,2130000,0.074921,4935932.0,0.045717,2.317339,3.787512,0.431529,mean


In [23]:
opt_df['spend'].sum(), nonopt_df['spend'].sum(), opt_df['incremental_outcome'].sum(), nonopt_df['incremental_outcome'].sum()

(np.int64(19900000),
 np.int64(28430000),
 np.float32(114760984.0),
 np.float32(233799800.0))

In [17]:
opt_df['incremental_outcome'].sum() / nonopt_df['incremental_outcome'].sum()

np.float32(0.4908515)

In [22]:
planner.input_data.media_spend.sel(time=slice(optimization_config['start_date'], optimization_config['end_date'])).sum(dim=('geo', 'time')).values.sum()

np.float64(28426286.546174083)

In [10]:
opt_attrs

{'start_date': '2024-07-06',
 'end_date': '2025-06-28',
 'budget': np.float32(19900000.0),
 'profit': np.float32(94860984.0),
 'total_incremental_outcome': np.float32(114760984.0),
 'total_roi': np.float32(5.766884),
 'total_cpik': np.float32(0.17340387),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [11]:
opt_results.nonoptimized_data.sel(metric='mean').attrs

{'start_date': '2024-07-06',
 'end_date': '2025-06-28',
 'budget': np.float32(28430000.0),
 'profit': np.float32(205369810.0),
 'total_incremental_outcome': np.float32(233799800.0),
 'total_roi': np.float32(8.223701),
 'total_cpik': np.float32(0.121599756),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True}

In [17]:
spend_grid = opt_results.optimization_grid.spend_grid.values
spend_grid.shape

(819, 4)

In [18]:
spend_grid[0]

array([9550000., 6540000., 5820000., 4890000.])

In [19]:
spend_grid[-1]

array([17730000.,       nan,       nan,       nan])

In [53]:
(17730000 - 9550000) // 10**4

818

In [61]:
rounded_spend = np.round(np.linspace(9550000, 17730000, 10**4), -4)
rounded_spend

array([ 9550000.,  9550000.,  9550000., ..., 17730000., 17730000.,
       17730000.])

In [69]:
np.round(14000, -4)

np.int64(10000)

In [57]:
spend_grid[:, 0]

array([ 9550000.,  9560000.,  9570000.,  9580000.,  9590000.,  9600000.,
        9610000.,  9620000.,  9630000.,  9640000.,  9650000.,  9660000.,
        9670000.,  9680000.,  9690000.,  9700000.,  9710000.,  9720000.,
        9730000.,  9740000.,  9750000.,  9760000.,  9770000.,  9780000.,
        9790000.,  9800000.,  9810000.,  9820000.,  9830000.,  9840000.,
        9850000.,  9860000.,  9870000.,  9880000.,  9890000.,  9900000.,
        9910000.,  9920000.,  9930000.,  9940000.,  9950000.,  9960000.,
        9970000.,  9980000.,  9990000., 10000000., 10010000., 10020000.,
       10030000., 10040000., 10050000., 10060000., 10070000., 10080000.,
       10090000., 10100000., 10110000., 10120000., 10130000., 10140000.,
       10150000., 10160000., 10170000., 10180000., 10190000., 10200000.,
       10210000., 10220000., 10230000., 10240000., 10250000., 10260000.,
       10270000., 10280000., 10290000., 10300000., 10310000., 10320000.,
       10330000., 10340000., 10350000., 10360000., 

In [31]:
gtol = 0.0001
raw_media_spend = planner.input_data.media_spend
hist_spend = raw_media_spend. \
  sel(time=slice(optimization_config['start_date'], optimization_config['end_date'])). \
  sum(dim=('geo', 'time'))

hist_budget = hist_spend.values.sum()

In [30]:
38300000.0 - 31299926


7000074.0

In [32]:
hist_budget * gtol

np.float64(3129.992686242001)

1. Process DataSheet

In [5]:
planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
data = planner.build_input_data()

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
I0000 00:00:1758210144.494777 4831083 service.cc:148] XLA service 0x110921e50 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758210144.494812 4831083 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1758210144.502930 4831083 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` a

In [10]:
data.reach.shape if data.reach else None

In [11]:
data.frequency.shape if data.frequency else None

In [12]:
print(data.kpi.shape)  # (n_geos, n_times)
print(data.media.shape)  # (n_geos, n_times, n_media_channels)
print(data.geo.shape)  # (n_geos,)
print(data.time.shape)  # (n_times,)
print(data.population.shape)  # (n_geos,)

(1, 117)
(1, 117, 3)
(1,)
(117,)
(1,)


2. Process ParameterSheet

In [13]:
parameter_arrays = planner.get_processed_parameter_arrays()
for param_name, param_array in parameter_arrays.items():
  print(f"{param_name=}", "\n")
  print(param_array)
  print("\n")


param_name='alpha_m' 

<xarray.DataArray 'alpha_m' (media_channel: 3)> Size: 24B
array([0.98, 0.45, 0.68])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'


param_name='ec_m' 

<xarray.DataArray 'ec_m' (media_channel: 3)> Size: 24B
array([0.53, 0.88, 1.1 ])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'


param_name='slope_m' 

<xarray.DataArray 'slope_m' (media_channel: 3)> Size: 24B
array([ 2.47,  2.55, 11.09])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'




3. Process CoefficientsSheet

In [14]:
coefficient_arrays = planner.get_processed_coefficients_arrays()
coefficient_arrays['beta_gm']

<xarray.DataArray 'beta_gm' (geo: 1, media_channel: 3)> Size: 12B
array([[2.786987 , 4.7182198, 8.437376 ]], dtype=float32)
Coordinates:
  * geo            (geo) <U12 48B 'national_geo'
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'

In [15]:
coefficient_arrays['beta_gm'].shape

(1, 3)

In [16]:
coefficient_arrays['beta_grf'].shape if 'beta_grf' in coefficient_arrays else None

4. Create a PointInference Object

In [25]:
self = planner
parameter_arrays = self.get_processed_parameter_arrays()
coefficient_arrays = self.get_processed_coefficients_arrays()
input_data_obj = self.build_input_data()

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(


In [27]:
from meridian.planner import point_inference_data
point_data = point_inference_data.PointInferenceData(
        parameter_arrays, coefficient_arrays,
        input_data_obj=input_data_obj
      )

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


In [29]:
point_data.posterior

<xarray.Dataset> Size: 2kB
Dimensions:         (chain: 1, draw: 1, media_channel: 3, geo: 1, time: 117,
                     knots: 1)
Coordinates:
  * chain           (chain) int64 8B 0
  * draw            (draw) int64 8B 0
  * media_channel   (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * geo             (geo) <U12 48B 'national_geo'
  * time            (time) int64 936B 0 1 2 3 4 5 6 ... 111 112 113 114 115 116
  * knots           (knots) int64 8B 0
Data variables:
    alpha_m         (chain, draw, media_channel) float32 12B 0.98 0.45 0.68
    ec_m            (chain, draw, media_channel) float32 12B 0.53 0.88 1.1
    slope_m         (chain, draw, media_channel) float32 12B 2.47 2.55 11.09
    beta_gm         (chain, draw, geo, media_channel) float32 12B 2.787 ... 8...
    mu_t            (chain, draw, time) float32 468B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    knot_values     (chain, draw, knots) float32 4B 0.0
    tau_g           (chain, draw, geo) float32 4B 0.0
    roi_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    mroi_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    contribution_m  (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    beta_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    eta_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0

In [30]:
inference_data = point_data.get_inference_data()
inference_data

Inference data with groups:
	> posterior
	> sample_stats

In [18]:
point_inference_data = planner.get_inference_data()
point_inference_data


/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


Inference data with groups:
	> posterior
	> sample_stats

5. Scaling InputData & Borrow other utils from Meridian

In [17]:
model_obj = model.Meridian(
    input_data=data,
    model_spec=spec.ModelSpec(knots=1),
    inference_data=point_inference_data
)
model_obj.sample_prior(n_draws=100, seed=42)

In [18]:
model_obj.kpi_transformer.population_scaled_mean

<tf.Tensor: shape=(), dtype=float32, numpy=83.07044982910156>

In [19]:
model_obj.kpi_transformer.population_scaled_stdev

<tf.Tensor: shape=(), dtype=float32, numpy=2.72761607170105>

In [20]:
model_obj.kpi_transformer._population

<tf.Tensor: shape=(2,), dtype=float32, numpy=array([1., 1.], dtype=float32)>

In [21]:
model_obj.kpi_scaled

<tf.Tensor: shape=(2, 117), dtype=float32, numpy=
array([[-0.9060734 ,  0.46627533,  0.03300569, -0.9011113 , -0.5526803 ,
         0.38676238, -0.2438645 , -1.4485806 , -1.231364  ,  0.15669592,
         2.1709857 ,  1.1342238 ,  0.18611014, -0.20842256, -0.88179463,
        -0.46724033,  0.3529539 ,  1.1400726 ,  0.33667484,  0.633575  ,
        -0.9652011 , -1.1212033 , -2.166916  , -0.9266264 , -1.3373655 ,
        -0.5102764 ,  0.42713282, -0.3856743 , -0.82637304,  0.42950755,
         0.57224596,  0.03648807, -1.3894893 ,  1.0525823 ,  1.7958425 ,
         2.4891634 , -0.10680418,  0.32406834,  0.8096632 , -0.3416033 ,
        -0.4761127 , -0.2677461 , -0.5219151 , -1.1787535 , -0.3700861 ,
        -0.62953323, -0.72947335, -1.9166825 , -1.1420753 , -0.79873216,
        -0.7493495 , -0.201841  ,  1.173909  ,  1.0577765 ,  0.6727483 ,
        -0.01989292,  0.6685191 ,  1.5287145 ,  1.6319916 ,  1.2698632 ,
         0.6388783 ,  0.5989023 , -0.20210952, -0.21623483, -1.3019067 ,
 

In [22]:
model_obj.media_tensors.media_transformer._scale_factors_gm

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[20.135593, 19.933535, 20.265293],
       [20.135593, 19.933535, 20.265293]], dtype=float32)>

In [23]:
model_obj.media_tensors.media_scaled

<tf.Tensor: shape=(2, 117, 3), dtype=float32, numpy=
array([[[0.9661663 , 0.9839545 , 1.0265459 ],
        [0.9207507 , 1.0689793 , 1.0347117 ],
        [0.9439883 , 0.99400026, 0.92758363],
        [0.9743503 , 0.95752794, 0.99221045],
        [1.1098284 , 1.0319173 , 0.9882946 ],
        [0.97872615, 1.1195135 , 0.9980713 ],
        [1.2066426 , 1.        , 0.9040634 ],
        [0.9300858 , 1.0040438 , 0.9594169 ],
        [0.9223113 , 0.91794294, 0.9970648 ],
        [1.0760689 , 1.087007  , 1.0761653 ],
        [0.8986156 , 1.0708874 , 1.0052425 ],
        [1.0198061 , 1.0275176 , 1.0337423 ],
        [0.9456945 , 0.9516925 , 1.0237765 ],
        [0.93098086, 1.0332588 , 0.9595538 ],
        [1.0456626 , 1.017292  , 1.0387762 ],
        [0.9217098 , 0.998143  , 1.0273719 ],
        [0.98610336, 1.1216959 , 1.1079466 ],
        [1.0924193 , 0.95500976, 1.0362443 ],
        [0.9676611 , 1.1028439 , 0.9660532 ],
        [0.9628795 , 1.0166713 , 1.017081  ],
        [0.999517  , 1.0386

In [24]:
model_obj.inference_data.posterior

<xarray.Dataset> Size: 2kB
Dimensions:         (chain: 1, draw: 1, media_channel: 3, geo: 2, time: 117,
                     knots: 1)
Coordinates:
  * chain           (chain) int64 8B 0
  * draw            (draw) int64 8B 0
  * media_channel   (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * geo             (geo) <U4 32B 'geo0' 'geo1'
  * time            (time) int64 936B 0 1 2 3 4 5 6 ... 111 112 113 114 115 116
  * knots           (knots) int64 8B 0
Data variables:
    alpha_m         (chain, draw, media_channel) float32 12B 0.99 0.38 0.69
    ec_m            (chain, draw, media_channel) float32 12B 0.55 1.16 1.11
    slope_m         (chain, draw, media_channel) float32 12B 2.36 2.52 12.46
    beta_gm         (chain, draw, geo, media_channel) float32 24B 3.738 ... 8...
    mu_t            (chain, draw, time) float32 468B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    knot_values     (chain, draw, knots) float32 4B 0.0
    tau_g           (chain, draw, geo) float32 8B 0.0 0.0
    roi_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    mroi_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    contribution_m  (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    beta_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    eta_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0

In [25]:
model_obj.kpi_transformer.population_scaled_mean, model_obj.kpi_transformer.population_scaled_stdev

(<tf.Tensor: shape=(), dtype=float32, numpy=83.07044982910156>,
 <tf.Tensor: shape=(), dtype=float32, numpy=2.72761607170105>)

6. Optimization

a) Scenario 1: Optimize Historical Budget

In [26]:
optimization_config = {
  'fixed_budget': True,

  # spend constraints
  'spend_constraint_lower': {  # (1 - value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  # 'Channel3': 0.3
  },

  'spend_constraint_upper': {  # (1 + value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  # 'Channel3': 0.3
  },

  # # optimization period
  # 'start_date': '2023-01-02',
  # 'end_date': '2024-01-01'
  'start_date': '2025-01-25',
  'end_date': '2025-03-29'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-09-17 15:12:10.903236: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,29,0.237705,137.630737,0.474589,4.745887,3.854686,0.210709,mean
1,Channel1,41,0.336066,174.007751,0.424409,4.244092,6.135113,0.235622,mean
2,Channel2,52,0.426230,384.518280,0.739458,7.394582,13.411420,0.135234,mean


In [27]:
opt_attrs

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(122.0),
 'profit': np.float32(574.15674),
 'total_incremental_outcome': np.float32(696.15674),
 'total_roi': np.float32(5.706203),
 'total_cpik': np.float32(0.1752479),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [28]:
opt_results.optimization_grid.spend_grid

<xarray.DataArray 'spend_grid' (grid_spend_index: 25, channel: 3)> Size: 600B
array([[29., 29., 28.],
       [30., 30., 29.],
       [31., 31., 30.],
       [32., 32., 31.],
       [33., 33., 32.],
       [34., 34., 33.],
       [35., 35., 34.],
       [36., 36., 35.],
       [37., 37., 36.],
       [38., 38., 37.],
       [39., 39., 38.],
       [40., 40., 39.],
       [41., 41., 40.],
       [42., 42., 41.],
       [43., 43., 42.],
       [44., 44., 43.],
       [45., 45., 44.],
       [46., 46., 45.],
       [47., 47., 46.],
       [48., 48., 47.],
       [49., 49., 48.],
       [50., 50., 49.],
       [51., 51., 50.],
       [52., 52., 51.],
       [53., 53., 52.]])
Coordinates:
  * grid_spend_index  (grid_spend_index) int64 200B 0 1 2 3 4 ... 20 21 22 23 24
  * channel           (channel) object 24B 'Channel0' 'Channel1' 'Channel2'

In [29]:
opt_results.optimization_grid.incremental_outcome_grid

<xarray.DataArray 'incremental_outcome_grid' (grid_spend_index: 25, channel: 3)> Size: 600B
array([[137.6307373 ,  96.59674072,   1.18696594],
       [141.40509033, 103.04357147,   1.83509064],
       [144.95904541, 109.54502106,   2.79328918],
       [148.30335999, 116.08286285,   4.18904877],
       [151.44900513, 122.63978577,   6.19217682],
       [154.40701294, 129.19921875,   9.02423859],
       [157.18807983, 135.74551392,  12.96646881],
       [159.80282593, 142.26422119,  18.36399078],
       [162.26138306, 148.74176025,  25.62220001],
       [164.57345581, 155.16567993,  35.18986511],
       [166.74835205, 161.5246582 ,  47.52278137],
       [168.79486084, 167.80844116,  63.02471161],
       [170.72129822, 174.00775146,  81.96676636],
       [172.53546143, 180.11448669, 104.39807129],
       [174.24468994, 186.1214447 , 130.06994629],
       [175.85583496, 192.0223999 , 158.40396118],
       [177.37532043, 197.81202698, 188.52545166],
       [178.80912781, 203.48596191, 219.36526489],
       [180.16290283, 209.04046631, 249.80783081],
       [181.44180298, 214.47271729, 278.84146118],
       [182.6506958 , 219.78036499, 305.67376709],
       [183.79412842, 224.96200562, 329.78890991],
       [184.8762207 , 230.0165863 , 350.94604492],
       [185.90090942, 234.943573  , 369.13604736],
       [186.87193298, 239.74301147, 384.51828003]])
Coordinates:
  * grid_spend_index  (grid_spend_index) int64 200B 0 1 2 3 4 ... 20 21 22 23 24
  * channel           (channel) object 24B 'Channel0' 'Channel1' 'Channel2'

In [30]:
data.kpi[:, -10:].sum()

<xarray.DataArray 'kpi' ()> Size: 8B
array(1659.426755)

In [31]:
opt_results.nonoptimized_data.sel(metric='mean').to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,41,0.336066,170.721298,0.416393,4.163934,1.846165,0.240157,mean
1,Channel1,41,0.336066,174.007751,0.424409,4.244092,6.135113,0.235622,mean
2,Channel2,40,0.327869,81.966766,0.204917,2.049169,21.401102,0.488003,mean


In [32]:
opt_attrs

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(122.0),
 'profit': np.float32(574.15674),
 'total_incremental_outcome': np.float32(696.15674),
 'total_roi': np.float32(5.706203),
 'total_cpik': np.float32(0.1752479),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [33]:
opt_results.nonoptimized_data.sel(metric='mean').attrs

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(122.0),
 'profit': np.float32(304.69583),
 'total_incremental_outcome': np.float32(426.69583),
 'total_roi': np.float32(3.4975069),
 'total_cpik': np.float32(0.28591797),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True}

In [42]:
display(opt_results.plot_response_curves())
display(opt_results.plot_incremental_outcome_delta())
display(opt_results.plot_spend_delta())
opt_attrs

alt.FacetChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

{'start_date': '2024-07-06',
 'end_date': '2025-06-28',
 'budget': np.float32(19900000.0),
 'profit': np.float32(94860984.0),
 'total_incremental_outcome': np.float32(114760984.0),
 'total_roi': np.float32(5.766884),
 'total_cpik': np.float32(0.17340387),
 'is_revenue_kpi': False,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [37]:
696.15674 / 1664


0.4183634254807692

b) Scenario 2: Increment Historical Budget By 20%

In [ ]:
optimization_config = {
  'fixed_budget': True,
  'budget': 1.2 * 31310000.0,  # 20% increase in historical budget

  # spend constraints
  'spend_constraint_lower': {  # (1 - value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  },

  'spend_constraint_upper': {  # (1 + value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  },

  # optimization period
  'start_date': '2023-01-02',
  'end_date': '2024-01-01'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,12290000,0.327123,20708256.0,0.018733,1.684968,1.057250,0.593483,mean
1,Channel1,12310000,0.327655,26938932.0,0.025021,2.188378,1.054687,0.456959,mean
2,Channel2,12970000,0.345222,33598688.0,0.030936,2.590492,1.083608,0.386027,mean


In [55]:
display(opt_results.plot_response_curves())
display(opt_results.plot_incremental_outcome_delta())
display(opt_results.plot_spend_delta())
opt_attrs

alt.FacetChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

{'start_date': '2023-01-02',
 'end_date': '2024-01-01',
 'budget': np.float32(37570000.0),
 'profit': np.float32(43675870.0),
 'total_incremental_outcome': np.float32(81245870.0),
 'total_roi': np.float32(2.16252),
 'total_cpik': np.float32(0.4624235),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': np.False_,
 'fixed_budget': True}

c) Scenario 3: Flex Budget Target ROI

In [ ]:
optimization_config = {
  'fixed_budget': False,
  'target_roi': 1.2,

  # optimization period
  'start_date': '2023-01-02',
  'end_date': '2024-01-01'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,27280000,0.435644,31709088.0,0.012923,1.162357,0.505293,0.860321,mean
1,Channel1,18700000,0.298627,32320182.0,0.019761,1.728352,0.666096,0.578586,mean
2,Channel2,16640000,0.265730,37058868.0,0.026596,2.227095,0.811803,0.449015,mean


In [57]:
optimization_config = {
  'fixed_budget': False,
  'target_roi': 2.0,

  # optimization period
  'start_date': '2023-01-02',
  'end_date': '2024-01-01'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,14660000,0.335930,23057266.0,0.017486,1.572801,0.921951,0.635808,mean
1,Channel1,14020000,0.321265,28633604.0,0.023351,2.042340,0.921755,0.489634,mean
2,Channel2,14960000,0.342805,35597744.0,0.028417,2.379528,0.921096,0.420251,mean


In [58]:
display(opt_results.plot_response_curves())
display(opt_results.plot_incremental_outcome_delta())
display(opt_results.plot_spend_delta())
opt_attrs

alt.FacetChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

{'start_date': '2023-01-02',
 'end_date': '2024-01-01',
 'budget': np.float32(43640000.0),
 'profit': np.float32(43648616.0),
 'total_incremental_outcome': np.float32(87288616.0),
 'total_roi': np.float32(2.0001974),
 'total_cpik': np.float32(0.49995065),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': False,
 'target_roi': 2}

In [59]:
opt_results.spend_bounds

(array([0.]), array([2.]))

In [62]:
nonopt_df = opt_results.nonoptimized_data.sel(metric='mean').to_dataframe().reset_index()
opt_df = opt_results.optimized_data.sel(metric='mean').to_dataframe().reset_index()


In [63]:
nonopt_df

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,13640000,0.435644,22085456.0,0.018002,1.619168,0.976730,0.617601,mean
1,Channel1,9350000,0.298627,23369356.0,0.028577,2.499396,1.366845,0.400097,mean
2,Channel2,8320000,0.265730,27261440.0,0.039130,3.276615,1.701346,0.305193,mean


In [64]:
opt_df

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,14660000,0.335930,23057266.0,0.017486,1.572801,0.921951,0.635808,mean
1,Channel1,14020000,0.321265,28633604.0,0.023351,2.042340,0.921755,0.489634,mean
2,Channel2,14960000,0.342805,35597744.0,0.028417,2.379528,0.921096,0.420251,mean


In [65]:
opt_df['spend'] / nonopt_df['spend']

0    1.074780
1    1.499465
2    1.798077
Name: spend, dtype: float64

In [66]:
opt_df['incremental_outcome'] / nonopt_df['incremental_outcome']

0    1.044002
1    1.225263
2    1.305791
Name: incremental_outcome, dtype: float32

In [ ]:
# # Configuration for input excel file
# input_config = {

#   # time and geo inputs
#   'time_col': 'week',
#   'geo_col': 'geo',
#   'population_col': 'population',

#   # kpi inputs
#   'kpi_col': 'conversions',  #
#   'kpi_type': 'non_revenue',
#   'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

#   # impression based media inputs
#   'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
#   'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
#   'media_channels': ['Channel0', 'Channel1', 'Channel2'],

#   # reach based media inputs
#   'reach_cols': ['Channel3_reach'],
#   'frequency_cols': ['Channel3_frequency'],
#   'rf_spend_cols': ['Channel3_spend'],
#   'rf_channels': ['Channel3'],


#   }


In [22]:
cost = np.array([72705368, 10431875, 34902239])
cost / cost.mean()

array([1.84782329, 0.26512845, 0.88704826])